# Bayesian Optimization for Black-Box Functions (Submission 11)

This notebook implements  **"Deep Recovery"** Bayesian Optimization pipeline configured for **Round 11 Queries** across all 8 independent black-box target functions.

### Core Strategic Configurations for Round 11:
* **Deep Acquisition Minimization (300 Multi-Starts):** To guarantee deep search resolution and avoid getting pinned in local landscape artifacts, local optimization steps execute **300 random multi-starts** per function.
* **"The Deep Recovery" $\xi$-Mapping Architecture:**
  * **$\xi = 0.5000$ (Max Discovery & Exploration Shock):** Retained for Channels 1, 3, 4, and 6 to continuously shock the optimization framework out of flat or stagnant dead-zones.
  * **$\xi = 0.5000$ (EMERGENCY RESET):** Applied critically to **Function 5**. Jitter is forcefully increased from 0.0001 up to 0.5000 to shock the model out of the local 16.84 trap and force it to explore and locate the highly lucrative 1593.32 historical region again.
  * **$\xi = 0.1000$ (Noisy Landscape Tracking):** Maintained for Channel 2 to regularize the broad optimization search over stochastic variance.
  * **$\xi = 0.00001$ (Surgical Peak Refinement):** Retained for Channel 7 to perform narrow micro-stepping adjustments directly on top of the 2.55 peak apex.
  * **$\xi = 0.0000$ (Pure Maximization Exploitation):** Dedicated to Channel 8 to track absolute local gradients and preserve structural placement right on the 9.9869 summit peak.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Enforce inline visualization support
%matplotlib inline 

print("Environment initialized. Round 11 Deep Recovery pipeline active.")

## Definition of Object-Oriented Bayesian Optimizer

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Initializes noise variance mappings.
        """
        # Using 1e-2 for F2 to handle stochastic noise, 1e-6 for others to maintain high fidelity
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads cumulative multi-round datasets and fits a precise GPR model.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        # Matern 2.5 kernel to perfectly balance smoothness and localized flexibility
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            nu=2.5
        )
        
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha,
            normalize_y=True, 
            n_restarts_optimizer=50 
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi):
        """
        Computes Expected Improvement (EI) surfaces for specified jitter configurations.
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi):
        """
        Executes 300 multi-start restarts across the L-BFGS-B optimizer bound space.
        """
        best_x = None
        max_ei = -1
        
        for _ in range(300):
            x0 = np.random.uniform(0.0, 1.0, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi),
                x0,
                bounds=[(0.0, 1.0)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Sequential Execution Optimization Pipeline

In [ ]:
output_file = "submission_11_results.txt"

# xi_map Logic: "The Deep Recovery"
# F1, F3, F4, F6: Continued max exploration (0.5) to break out of stagnant/negative zones.
# F5: EMERGENCY RESET. Increased from 0.0001 to 0.5 to force the model to look away 
#     from the local 16.84 trap and find the 1593.32 region again.
# F7: Surgical exploitation (0.00001) to stay on the 2.55 peak.
# F8: Pure exploitation (0.0) with high restarts to stay near 9.9869.
xi_map = {
    1: 0.5000, 
    2: 0.1000, 
    3: 0.5000, 
    4: 0.5000, 
    5: 0.5000, # Shock jitter introduced for recovery
    6: 0.5000,
    7: 0.00001,
    8: 0.0000
}

print("Generating Submission 11: The Deep Recovery...")
print("-" * 55)

notebook_results = {}

for i in range(1, 8 + 1):
    func_dir = f"function_{i}"
    
    if not os.path.exists(func_dir):
        print(f"Warning: Functional target context '{func_dir}' missing from directory path.")
        continue
        
    optimizer = BayesianOptimizer(is_noisy=(i == 2))
    optimizer.load_and_fit(func_dir)
    
    current_xi = xi_map[i]
    next_point = optimizer.propose_next_point(xi=current_xi)
    formatted_point = "-".join([f"{val:.6f}" for val in next_point])
    
    notebook_results[f"Function {i}"] = (formatted_point, current_xi)
    print(f"Function {i}: {formatted_point} (xi={current_xi})")

# Write finalized coordinates to target result manifest
with open(output_file, "w") as f:
    for i in range(1, 9):
        key = f"Function {i}"
        if key in notebook_results:
            f.write(f"{key}: {notebook_results[key][0]}\n")

print("-" * 55)
print(f"Submission 11 successfully written onto file: {output_file}")

### Summary Evaluation Matrix

In [ ]:
print(f"| {'Target Channel':15} | {'Exploration (xi)':18} | {'Proposed Query Coordinates (Round 11)':55} |")
print(f"| {'-'*15} | {'-'*18} | {'-'*55} |")
for func, (coords, xi_val) in notebook_results.items():
    print(f"| {func:15} | {xi_val:<18} | {coords:55} |")